# Foundation-Sec-8B QLoRA — Kaggle (free T4)

Fine-tunes **fdtn-ai/Foundation-Sec-8B-Instruct** on `data/train.jsonl`, then exports a
merged **GGUF (Q4_K_M)** + Ollama Modelfile to run locally on an 8 GB GPU.

**Before running:** enable *GPU T4* and *Internet* in the Kaggle sidebar, and add `data/`
as a Kaggle Dataset input. Run with `SMOKE = True` once to validate the whole path.

In [ ]:
SMOKE = True            # True = 50 rows / 10 steps (validate); False = full run
BASE_MODEL = "fdtn-ai/Foundation-Sec-8B-Instruct"
MAX_SEQ_LEN = 2048
SEED = 42
OUT_GGUF = "gguf_model"

In [ ]:
# Kaggle internet must be ON (sidebar). Unsloth pulls transformers/trl/peft/bitsandbytes.
!pip install -q unsloth

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL, max_seq_length=MAX_SEQ_LEN, load_in_4bit=True, dtype=None,
)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model, r=32, lora_alpha=32, lora_dropout=0,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    use_gradient_checkpointing="unsloth", random_state=SEED,
)

In [ ]:
from datasets import load_dataset

ds = load_dataset("json", data_files={"train": "data/train.jsonl", "eval": "data/eval.jsonl"})

def to_text(ex):
    # The chat template MUST match the base model (Llama-3.1); prepare_data already
    # normalized every row to it, so this is the one place it is applied.
    return {"text": tokenizer.apply_chat_template(ex["messages"], tokenize=False)}

ds = ds.map(to_text)
if SMOKE:
    ds["train"] = ds["train"].select(range(min(50, len(ds["train"]))))
print(ds)

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth import is_bfloat16_supported

args = SFTConfig(
    output_dir="out", dataset_text_field="text", max_seq_length=MAX_SEQ_LEN,
    per_device_train_batch_size=2, gradient_accumulation_steps=8,
    learning_rate=2e-4, lr_scheduler_type="cosine", warmup_ratio=0.05,
    num_train_epochs=1, max_steps=(10 if SMOKE else -1),
    logging_steps=5, optim="paged_adamw_8bit", seed=SEED, fp16=not is_bfloat16_supported(), bf16=is_bfloat16_supported(),
)
trainer = SFTTrainer(model=model, tokenizer=tokenizer, train_dataset=ds["train"], args=args)
trainer.train()

In [ ]:
# Quick held-out sanity check (not a full eval — see DATA_PLAN.md).
print(trainer.evaluate(eval_dataset=ds["eval"]))

## Export — merge BEFORE GGUF

Unsloth issue **#11698**: exporting to GGUF *without merging first* silently ships the
base model and throws your fine-tune away. So we merge explicitly, then quantize. On a
16 GB T4 the fp16 merge is tight — Kaggle's ~30 GB system RAM absorbs it.

In [ ]:
# 1) merge the LoRA adapter into the base weights (explicit, before any export)
model.save_pretrained_merged("merged_16bit", tokenizer, save_method="merged_16bit")

# 2) quantize the merged model to GGUF Q4_K_M (fits an 8 GB RTX 4060 locally)
model.save_pretrained_gguf(OUT_GGUF, tokenizer, quantization_method="q4_k_m")

In [ ]:
import glob, os
gguf = sorted(glob.glob(f"{OUT_GGUF}/*.gguf"))[0]
print("GGUF:", gguf)
# Unsloth emits an Ollama Modelfile with the chat template baked in; confirm it exists.
print("Modelfile present:", os.path.exists(f"{OUT_GGUF}/Modelfile"))
print("\nDownload the .gguf + Modelfile, then locally:")
print("  ollama create cyber-sec -f Modelfile && ollama run cyber-sec")

Only the ~5 GB GGUF comes down to your machine. Everything above ran on Kaggle's network.